In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



In [4]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,Dropout,BatchNormalization,GlobalAveragePooling2D

from keras.layers import RandomFlip, RandomRotation, RandomZoom, Input
from keras.callbacks import EarlyStopping,ModelCheckpoint



In [5]:
df = pd.read_csv('./data/fer2013.csv')

In [6]:
df.head()

,emotion,pixels,Usage
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...,Training
1,0,151 150 147 155 148 133 111 140 170 174 182 15...,Training
2,2,231 212 156 164 174 138 161 173 182 200 106 38...,Training
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...,Training
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...,Training


In [5]:
#0: Angry
#1: Disgust
#2: Fear
#3: Happy
#4: Sad
#5: Surprise
#6: Neutral

In [7]:
df['emotion'].value_counts()

emotion
3    8989
6    6198
4    6077
2    5121
0    4953
5    4002
1     547
Name: count, dtype: int64

In [ ]:
train_data = df[df['Usage'] == 'Training']
val_data = df[df['Usage'] == 'PublicTest']
test_data = df[df['Usage'] == 'PrivateTest']


def pixel_preprocess(df):


    x=[]
    y=[]

    for index,row in df.iterrows():  # iterating over all rows
        pixels = np.array(row['pixels'].split(),dtype='float32')
        pixels = pixels.reshape(48,48,1)
        x.append(pixels)
        y.append(row['emotion'])

    x = np.array(x)  
    y = keras.utils.to_categorical(np.array(y), num_classes=7)
    return x,y

X_train,y_train = pixel_preprocess(train_data)
X_val,y_val = pixel_preprocess(val_data)
X_test,y_test = pixel_preprocess(test_data)


print("Train",X_train.shape,y_train.shape)
print("Val",X_val.shape,y_val.shape)
print("Test",X_test.shape,y_test.shape)






Train (28709, 48, 48, 1) (28709, 7)
Val (3589, 48, 48, 1) (3589, 7)
Test (3589, 48, 48, 1) (3589, 7)


In [9]:
X_train = X_train / 255.0
X_val   = X_val / 255.0
X_test  = X_test / 255.0

In [24]:
model=Sequential()

model.add(Input(shape=(48,48,1)))


model.add(Conv2D(32,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(32,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=1,padding="same"))
model.add(Dropout(0.25))

model.add(Conv2D(64,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(64,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=1,padding='same'))
model.add(Dropout(0.25))

model.add(Conv2D(64,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(64,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=1,padding='same'))


model.add(Dropout(0.3))
model.add(Conv2D(64,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(64,kernel_size=(3,3),padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=1,padding='same'))
model.add(Dropout(0.3))


model.add(GlobalAveragePooling2D())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(7,activation='softmax'))


In [17]:
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
checkpoint = ModelCheckpoint('emotion_model.h5',monitor='val_loss',save_best_only=True)

In [18]:
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_6 (Conv2D)           (None, 48, 48, 32)        320       
                                                                 
 batch_normalization_6 (Batc  (None, 48, 48, 32)       128       
 hNormalization)                                                 
                                                                 
 conv2d_7 (Conv2D)           (None, 48, 48, 32)        9248      
                                                                 
 batch_normalization_7 (Batc  (None, 48, 48, 32)       128       
 hNormalization)                                                 
                                                                 
 max_pooling2d_3 (MaxPooling  (None, 48, 48, 32)       0         
 2D)                                                             
                                                      

In [23]:
optimizer = keras.optimizers.Adam(learning_rate=0.0003)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [22]:
history = model.fit(
    X_train,y_train,
    epochs=60,
    batch_size=64,
    shuffle=True,
    validation_data=(X_val,y_val),
    callbacks=[early_stop,checkpoint]
    )


RuntimeError: You must compile your model before training/testing. Use `model.compile(optimizer, loss)`.

In [15]:
emotion_labels = [
    "Angry", 
    "Disgust", 
    "Fear", 
    "Happy", 
    "Sad", 
    "Surprise", 
    "Neutral"
]

In [17]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("emotion_model.h5")

emotion_labels = [
    "Angry", 
    "Disgust", 
    "Fear", 
    "Happy", 
    "Sad", 
    "Surprise", 
    "Neutral"
]

# Load face detector (Haar cascade)
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]

        # Resize to 48x48 (same as training)
        face = cv2.resize(face, (48, 48))

        # Normalize like training
        face = face / 255.0

        # Reshape to model input
        face = np.reshape(face, (1, 48, 48, 1))

        # Predict
        predictions = model.predict(face, verbose=0)
        emotion = emotion_labels[np.argmax(predictions)]

        # Draw rectangle + label
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, emotion, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, (0,255,0), 2)

    cv2.imshow("Emotion Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
